# v1.6 pages-focus run (A100, ~14-16h, resumable)

Continues from the v1.5 final adapter with the decontaminated mixture: the
fixed-percentage rashi/tosafot crops captured the marginal *apparatus*, not
the commentary (no vertical strip can contain the L-shaped Vilna
commentaries), so v1.6 drops them entirely and trains small-script reading on
**full-page extraction** instead. Mixture: 35% small-script page rows / 30%
synthetic v2 (18-48px renders) / 25% gemara crops / 10% gemara page rows
(seesaw replay).

**Resolution policy (global, ships with the model):** `min_pixels=6.5MP`.
Pages are ~4MP native (rashi glyphs ~15px), so training upscales them to
~6.5MP (~19px glyphs, near the crop regime). The policy is global rather than
per-domain because the exported `preprocessor_config` is the ONLY resolution
contract at inference (the LM Studio harness sends native-res images) — a
per-domain training policy could not be reproduced at eval time, and
train/infer resolution mismatch is proven harmful.

Goal: page-level rashi CER 0.362 → 0.15-0.25 (Gemini Pro: 0.315).
Eval of the merged model ONLY via the LM Studio API harness.
Secrets: `HF_TOKEN`, `WANDB_API_KEY`.

In [ ]:
# Cell 1 — installs + env
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["WANDB_PROJECT"] = "qwen-hebrew-finetune"
%pip install -q "unsloth[colab-new]" hf_transfer wandb

import torch
assert torch.cuda.is_available(), "No GPU — switch runtime to A100"

In [ ]:
# Cell 2 — data: talmud v2 (all-pages, gemara-only crops) + synthetic v2
from google.colab import userdata
from huggingface_hub import login, snapshot_download
from datasets import load_dataset

login(token=userdata.get("HF_TOKEN"))

TALMUD_REPO = "isaacmg/talmud_finetune_v2"
SYNTH_REPO = "isaacmg/synthetic_rashi"
SYNTH_REVISION = "b4e1254e1b51ff93958599ccd46ecf2992cd2e97"  # v2: 18-48px renders, seed 20260720

talmud = load_dataset(TALMUD_REPO, split="train")
talmud_val = load_dataset(TALMUD_REPO, split="val")
synth = load_dataset(SYNTH_REPO, split="train", revision=SYNTH_REVISION)
synth_eval = load_dataset(SYNTH_REPO, split="eval", revision=SYNTH_REVISION)

crops = talmud.filter(lambda t: t == "crop_transcribe", input_columns="task")
pages = talmud.filter(lambda t: t == "page_extract", input_columns="task")
small_script_crops = crops.filter(
    lambda s: s in ("rashi", "tosafot"), input_columns="section")
gemara_crops = crops.filter(lambda s: s == "gemara", input_columns="section")
pages_small = pages.filter(
    lambda s: s in ("rashi", "tosafot"), input_columns="section")
pages_gemara = pages.filter(lambda s: s == "gemara", input_columns="section")

print(f"synth={len(synth)} gemara_crops={len(gemara_crops)} "
      f"pages_small={len(pages_small)} pages_gemara={len(pages_gemara)}")
assert len(small_script_crops) == 0, (
    "contaminated rashi/tosafot crop rows present — wrong dataset version")
assert len(synth) > 18500 and len(gemara_crops) > 5000
assert len(pages_small) > 9800 and len(pages_gemara) > 5000

In [ ]:
# Cell 3 — model at PAGE RESOLUTION + v1.5 final adapter warm start (atomic cell)
from unsloth import FastVisionModel
from transformers import AutoImageProcessor

MAX_SEQ = 12288
# Global 6.5MP floor: pages (~4MP native) upscale to ~6.5MP (~19px rashi
# glyphs); crops/synth ride the same policy. MUST ship in the exported
# preprocessor_config — it is the only train/infer resolution contract.
MIN_PIX = 6_500_000
MAX_PIX = 7_000_000

V15_CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-rashi-ckpt"
V15_REVISION = "e050aca83e16d7386128283b7e9add98dcb7d012"  # v1.5 final, step 2000

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ,
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none", random_state=3407,
)

# weights-only warm start from the v1.5 final adapter (fresh optimizer + schedule)
assert len(V15_REVISION) == 40, "invalid revision SHA"
from safetensors.torch import load_file
from peft import set_peft_model_state_dict
local = snapshot_download(V15_CKPT_REPO, revision=V15_REVISION,
                          allow_patterns="last-checkpoint/adapter_model.safetensors")
missing = set_peft_model_state_dict(
    model, load_file(f"{local}/last-checkpoint/adapter_model.safetensors"))
print("unexpected keys:", len(getattr(missing, "unexpected_keys", [])))

tokenizer.image_processor = AutoImageProcessor.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct", min_pixels=MIN_PIX, max_pixels=MAX_PIX,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
lora_params = [n for n, p in model.named_parameters()
               if p.requires_grad and "lora" in n.lower()]
assert trainable > 0 and lora_params
print(f"trainable: {trainable/1e6:.1f}M ({len(lora_params)} LoRA tensors)")
print("resolution:", tokenizer.image_processor.size)

In [ ]:
# Cell 4 — conversation format + collator (native res preserved by resize='max')
def to_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text", "text": sample["question"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["answer"]},
            ]},
        ]
    }

from unsloth.trainer import UnslothVisionDataCollator

collator = UnslothVisionDataCollator(
    model, tokenizer,
    formatting_func=to_conversation,
    resize="max",
    max_seq_length=MAX_SEQ,
    train_on_responses_only=True,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# guardrail: one real batch must show page-res pixels + masked labels
batch = collator([pages_small[0], synth[0]])
pv = batch["pixel_values"]
assert pv is not None and pv.shape[0] > 40000, (
    f"{pv.shape[0]} patch rows — expected >40k for two ~6.5MP images; "
    "the resolution policy is not reaching the collator")
labels = batch["labels"]
unmasked = labels[0][labels[0] != -100]
_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
assert pages_small[0]["answer"][:30] in _tok.decode(unmasked)
print(f"collator OK: pixel rows={pv.shape[0]}, labels masked")

In [ ]:
# Cell 5 — mixture + per-domain eval + training (resumable, max_steps-bounded)
import inspect
import wandb
from datasets import concatenate_datasets, interleave_datasets
from huggingface_hub import list_repo_files
from trl import SFTConfig, SFTTrainer
from unsloth import is_bf16_supported

CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-v16-ckpt"  # private, auto-created
wandb.login(key=userdata.get("WANDB_API_KEY"))

# 35% small-script pages / 30% synth / 25% gemara crops / 10% gemara pages
mixture = interleave_datasets(
    [synth, pages_small, gemara_crops, pages_gemara],
    probabilities=[0.30, 0.35, 0.25, 0.10], seed=3407,
    stopping_strategy="all_exhausted",
)
# per-domain val LOSS every 100 steps: page rows + gemara crops + synth eval
eval_ds = concatenate_datasets([
    talmud_val.filter(lambda t: t == "page_extract", input_columns="task")
              .select(range(60)),
    talmud_val.filter(
        lambda t, s: t == "crop_transcribe" and s == "gemara",
        input_columns=["task", "section"]).select(range(20)),
    synth_eval.select(range(40)),
])

def make_sft_config(**kw):
    params = inspect.signature(SFTConfig.__init__).parameters
    if "max_seq_length" in kw and "max_seq_length" not in params:
        kw["max_length"] = kw.pop("max_seq_length")
    dropped = {k: kw.pop(k) for k in list(kw) if k not in params}
    if dropped:
        print(f"⚠️ dropped unsupported SFTConfig kwargs: {sorted(dropped)}")
    return SFTConfig(**kw)

def make_trainer(**kw):
    try:
        return SFTTrainer(**kw)
    except TypeError as e:
        if "tokenizer" in kw and ("tokenizer" in str(e) or "processing_class" in str(e)):
            kw["processing_class"] = kw.pop("tokenizer")
            return SFTTrainer(**kw)
        raise

resume_dir = None
try:
    # hub_strategy="checkpoint" pushes a rolling "last-checkpoint/" folder
    # (NOT numbered checkpoint-N folders).
    files = list_repo_files(CKPT_REPO)
    if any(f.startswith("last-checkpoint/") for f in files):
        snapshot_download(CKPT_REPO, allow_patterns="last-checkpoint/*",
                          local_dir="outputs_v16")
        resume_dir = "outputs_v16/last-checkpoint"
        print("resuming from last-checkpoint")
except Exception as e:
    print(f"no checkpoint repo yet ({type(e).__name__}) — fresh start")

FastVisionModel.for_training(model)
trainer = make_trainer(
    model=model, tokenizer=tokenizer, data_collator=collator,
    train_dataset=mixture, eval_dataset=eval_ds,
    args=make_sft_config(
        # batch 1 x accum 8: ~6.3k image tokens per 6.5MP sample.
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        max_steps=3000,                    # 24k samples ≈ 14h; sweep picks the step
        learning_rate=5e-5,                # continuation LR
        warmup_ratio=0.02, lr_scheduler_type="cosine", weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps", eval_steps=100, per_device_eval_batch_size=1,
        save_steps=100, save_total_limit=2,
        push_to_hub=True, hub_model_id=CKPT_REPO,
        hub_strategy="checkpoint", hub_private_repo=True,
        optim="adamw_8bit", seed=3407, output_dir="outputs_v16",
        report_to="wandb", run_name="pages_focus_v16",
        bf16=is_bf16_supported(), fp16=not is_bf16_supported(),
        remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=MAX_SEQ,
    ),
)
trainer.train(resume_from_checkpoint=resume_dir)

In [ ]:
# Cell 6 — export merged (policy-carrying) model
MERGED_REPO = "isaacmg/qwen3-vl-8b-hebrew-v16-merged"
model.save_pretrained_merged("v16-merged", tokenizer, save_method="merged_16bit")
# ship the TRAINING resolution policy with the model (hard rule)
tokenizer.image_processor.save_pretrained("v16-merged")
model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", private=True)
print("pushed", MERGED_REPO)